In [1]:
import plotly.express as px
import pandas as pd
# import humanize
import math
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import glob
import numpy as np

In [2]:
selected_columns = [
        'Year',
        'entry_cooloff',
        'exit_cooloff',
        'macd_period',
        'macd_variable',
        'Total P/L',
        'Avg. Monthly PnL',
        'Max Monthly Gain',
        'Max Monthly Loss',
        'SD Monthly PnL',
        'Monthly Avg. PnL / SD',
        'Peak Drawdown',
        'Ratio (Total P/L / Net Premium)',
        'transaction_fee_bt',
        # 'transaction_fee_blotter',
        'Daily Peak Margin',
        'Avg. Daily Peak Margin',
        # 'total_trade_duration', 'avg_duration_per_trade', 'pct_time_in_trade',
        # 'total_trades',
        # 'peak_contracts',
        # 'avg_active_contracts',
        # 'max_trade_count',
        # 'mean_trade_count',
        # 'min_trade_count',
        # 'max_negative_trades',
        # 'mean_negative_trades',
        # 'min_negative_trades',
        # 'max_positive_trades',
        # 'mean_positive_trades',
        # 'min_positive_trades',
        # # 'Avg. Daily PnL',
        # # 'Max Daily Gain',
        # # 'Max Daily Loss',
        # 'avg_lt_2min',
        # 'avg_lt_5min',
        # 'days_with_lt_2min',
        # 'days_with_lt_5min',
        # 'days_with_lt_2min_in_pct',
        # 'days_with_lt_5min_in_pct'
    ]

def excel_to_dict(file_path):
    result_dict = {}
    # Read Excel file into a pandas DataFrame
    df = pd.read_excel(file_path)
    # Assuming the first row is a header, we skip it
    for index, row in df.iterrows():
        for i in range(0, len(row), 2):
            if i + 1 < len(row):
                result_dict[row[i]] = row[i + 1]
    return result_dict

def convert_to_number(value):
    try:
        # Remove non-numeric characters and convert to float
        return float(value.replace(',', '').split()[0])
    except ValueError:
        return value  # Return the original value if conversion fails
    

def total_trades_folder(folder_path):
    blotter_path_list = glob.glob(folder_path + '/blotter/*.csv')
    blotter_df = pd.concat((pd.read_csv(one_blotter_path) for one_blotter_path in blotter_path_list))

    # filter only trades and negative position
    trade_df = blotter_df[(blotter_df['trade_sub_type'] == 'trade') & (blotter_df['position'] < 0)]

    # lot size / trade size
    single_trade_position = trade_df['position'].values[0]

    # dividing by 2 as put and call
    total_trades = trade_df['position'].sum() / (single_trade_position*2)

    # print(total_trades)
    return total_trades

def peak_contracts_per_folder(folder_path):
    blotter_path_list = glob.glob(folder_path + '/blotter/*.csv')
    full_blotter_df = pd.concat((pd.read_csv(one_blotter_path) for one_blotter_path in blotter_path_list))

    full_blotter_df['timestamp'] = pd.to_datetime(full_blotter_df['timestamp'])
    full_blotter_df.set_index('timestamp', inplace=True)
    full_blotter_df = full_blotter_df.sort_index()
    
    full_blotter_df['trade_position'] = np.where(full_blotter_df['trade_sub_type'] == 'trade', -full_blotter_df['position'], 0)
    full_blotter_df['cum_trade_position'] = full_blotter_df['trade_position'].cumsum()

    full_blotter_df['cum_trade_max'] = np.where(full_blotter_df['trade_sub_type'] == 'unwind', full_blotter_df['cum_trade_position'], np.nan)
    full_blotter_df['cum_trade_max'] = full_blotter_df['cum_trade_max'].ffill()
    full_blotter_df['cum_trade_max'] = full_blotter_df['cum_trade_max'].fillna(0)

    full_blotter_df['active_contracts'] = full_blotter_df['cum_trade_position'] - full_blotter_df['cum_trade_max']


    return full_blotter_df['active_contracts'].max() , full_blotter_df['active_contracts'].mean()


def count_weekends(start_date, end_date):
    # Generate all dates between start_date and end_date
    all_dates = np.arange(start_date, end_date + np.timedelta64(1, 'D'), dtype='datetime64[D]')
    days_of_week = (all_dates.astype('datetime64[D]') - np.datetime64('1970-01-05')).astype('timedelta64[D]').astype(int) % 7
    # Count Saturdays (5) and Sundays (6)
    weekends_count = np.isin(days_of_week, [5, 6]).sum()
    return weekends_count

# def time_in_trade(folder_path):
#     blotter_path_list = glob.glob(folder_path + '/blotter/*.csv')
#     full_blotter_df = pd.concat((pd.read_csv(one_blotter_path) for one_blotter_path in blotter_path_list))
#     full_blotter_df['date'] = pd.to_datetime(full_blotter_df['timestamp'])
#     entry_dates = full_blotter_df.loc[full_blotter_df['trade_sub_type'] == 'trade', 'date'].unique()
#     exit_dates = full_blotter_df.loc[full_blotter_df['trade_sub_type'] == 'unwind', 'date'].unique()

#     entry_dates.sort()
#     exit_dates.sort()

#     durations = [] 

#     for entry_date, exit_date in zip(entry_dates, exit_dates):
#         # Calculate total difference in days
#         entry_day = entry_date.astype('datetime64[D]')
#         exit_day = exit_date.astype('datetime64[D]')
#         total_days = (exit_day - entry_day).astype(int)
#         weekend_days = count_weekends(entry_date, exit_date) 

#         # Convert total days to minutes (18 hours per day = 1080 minutes)
#         minutes_from_days = total_days * 1065 

#         # Calculate total difference in minutes
#         total_minutes = (exit_date - entry_date) / np.timedelta64(1, 'm')

#         # Subtract minutes from days to get the remaining fractional minutes
#         fractional_minutes = total_minutes - minutes_from_days - weekend_days * 375

#         durations.append(fractional_minutes)

#     total_trade_duration = sum(durations)
#     avg_duration_per_trade =  sum(durations)/ len(durations)
#     pct_time_in_trade =  sum(durations)/(375*252) * 100

#     return (total_trade_duration, avg_duration_per_trade, pct_time_in_trade) 

def win_trades(folder_path):
    import glob
    import pandas as pd
    import numpy as np

    blotter_path_list = glob.glob(folder_path + '/blotter/*.csv')
    full_blotter_df = pd.concat((pd.read_csv(one_blotter_path) for one_blotter_path in blotter_path_list))
    full_blotter_df['date'] = pd.to_datetime(full_blotter_df['timestamp'])
    entry_dates = full_blotter_df.loc[full_blotter_df['trade_sub_type'] == 'trade', 'date'].unique()
    exit_dates = full_blotter_df.loc[full_blotter_df['trade_sub_type'] == 'unwind', 'date'].unique()

    entry_dates.sort()
    exit_dates.sort()

    cs_path_list = glob.glob(folder_path + '/consolidated_store/*.csv')
    full_cs_df = pd.concat((pd.read_csv(one_cs_path) for one_cs_path in cs_path_list))
    full_cs_df['date'] = pd.to_datetime(full_cs_df['timestamp'])
    full_cs_df = full_cs_df[full_cs_df['trade'] == True]


def trade_duration(folder_path):

    Blotter_df_list = glob.glob(folder_path + '/blotter/*.csv')
    Blotter_bt_df_list = [pd.read_csv(i) for i in Blotter_df_list]

    df = pd.concat(Blotter_bt_df_list, ignore_index=True)

    df['timestamp'] = pd.to_datetime(df['timestamp'])
    # Extract the trade date for grouping
    df['date'] = df['timestamp'].dt.date

    total_number_of_trading_days = df['date'].nunique()
    # print(f"{total_number_of_trading_days=}")

    # Filtering trades and unwinds separately
    trades = df[df['trade_sub_type'] == 'trade'][['date', 'timestamp', 'trade_sub_type']].drop_duplicates().sort_values('timestamp').reset_index(drop=True)
    unwinds = df[df['trade_sub_type'] == 'unwind'][['date', 'timestamp', 'trade_sub_type']].drop_duplicates().sort_values('timestamp').reset_index(drop=True)

    # # # Calculating duration
    trades['duration_minutes'] = (unwinds['timestamp'] - trades['timestamp']).dt.total_seconds()//60

    # Categorizing trades based on duration
    trades['lt_2min'] = trades['duration_minutes'] < 2
    trades['lt_5min'] = trades['duration_minutes'] < 5

    # Grouping by date
    daily_counts = trades.groupby('date').agg({'lt_2min': 'sum', 'lt_5min': 'sum'})

    # Compute daily averages
    avg_lt_2min = daily_counts['lt_2min'].mean()
    avg_lt_5min = daily_counts['lt_5min'].mean()

    # Count number of days with at least one trade in each category
    days_with_lt_2min = (daily_counts['lt_2min'] > 0).sum()
    days_with_lt_5min = (daily_counts['lt_5min'] > 0).sum()

    return round(avg_lt_2min,2), round(avg_lt_5min, 2), days_with_lt_2min, days_with_lt_5min, round(days_with_lt_2min/total_number_of_trading_days*100, 2), round(days_with_lt_5min/total_number_of_trading_days*100, 2)
    

def time_in_trade(folder_path):
    import glob
    import pandas as pd
    import numpy as np

    blotter_path_list = glob.glob(folder_path + '/blotter/*.csv')
    full_blotter_df = pd.concat((pd.read_csv(one_blotter_path) for one_blotter_path in blotter_path_list))
    full_blotter_df['date'] = pd.to_datetime(full_blotter_df['timestamp'])
    entry_dates = full_blotter_df.loc[full_blotter_df['trade_sub_type'] == 'trade', 'date'].unique()
    exit_dates = full_blotter_df.loc[full_blotter_df['trade_sub_type'] == 'unwind', 'date'].unique()

    entry_dates.sort()
    exit_dates.sort()

    durations = [] 

    for entry_date, exit_date in zip(entry_dates, exit_dates):
        # Calculate total difference in days
        entry_day = entry_date.astype('datetime64[D]')
        exit_day = exit_date.astype('datetime64[D]')
        total_days = (exit_day - entry_day).astype(int)
        weekend_days = count_weekends(entry_date, exit_date) 

        # Convert total days to minutes (18 hours per day = 1080 minutes)
        minutes_from_days = total_days * 1065 

        # Calculate total difference in minutes
        total_minutes = (exit_date - entry_date) / np.timedelta64(1, 'm')

        # Subtract minutes from days to get the remaining fractional minutes
        fractional_minutes = total_minutes - minutes_from_days - weekend_days * 375

        durations.append(fractional_minutes)

    total_trade_duration = sum(durations)
    avg_duration_per_trade = sum(durations) / len(durations)
    std_of_duration_per_trade = np.std(durations, ddof=0)  # Calculate standard deviation (sample std)

    pct_time_in_trade = sum(durations) / (375 * 252) * 100

    return total_trade_duration, avg_duration_per_trade, std_of_duration_per_trade, pct_time_in_trade

    
def get_transaction_fee(folder_path):
    try:
        Alpha_list = glob.glob(folder_path + '/consolidated_store/*.csv')
        Alpha_bt_df_list = [pd.read_csv(i) for i in Alpha_list]

        Alpha_df = pd.concat(Alpha_bt_df_list, ignore_index=True)
        Alpha_df = Alpha_df[Alpha_df['trade_done']==True]
        
        return Alpha_df['transaction_fees'].sum()
    except Exception as e:
        print(f"Error: {folder_path}")

def get_transaction_fee_blotter(folder_path):
    try:
        # print(folder_path)
        Blotter_df_list = glob.glob(folder_path + '/blotter/*.csv')
        Blotter_bt_df_list = [pd.read_csv(i) for i in Blotter_df_list]

        Blotter_df = pd.concat(Blotter_bt_df_list, ignore_index=True)
        
        return Blotter_df['exchange_fee'].sum()
    except Exception as e:
        print(f"Error: {folder_path}")


def get_trade_summary_stats(folder_path):
    """
    Returns summary statistics: max, mean, min for:
    - trade_count
    - negative_trades
    - positive_trades

    Parameters:
    df (pd.DataFrame): DataFrame with 'trade_sub_type' and 'position' columns, indexed by timestamp.

    Returns:
    dict: Dictionary containing 9 summary statistics values.
    """
    Blotter_df_list = glob.glob(folder_path + '/blotter/*.csv')
    Blotter_bt_df_list = [pd.read_csv(i) for i in Blotter_df_list]
    df = pd.concat(Blotter_bt_df_list, ignore_index=True)

    df.set_index('timestamp', inplace=True)

    # Ensure index is a DateTimeIndex
    df.index = pd.to_datetime(df.index)

    # Extract date from index
    df['date'] = df.index.date

    # Filter only rows where trade_sub_type == 'trade'
    trade_df = df[df['trade_sub_type'] == 'trade']

    # Group by date and calculate required metrics
    trade_stats = trade_df.groupby('date').agg(
        trade_count=('trade_sub_type', 'count'),
        negative_trades=('position', lambda x: (x < 0).sum()),
        positive_trades=('position', lambda x: (x > 0).sum())
    )

    trade_stats = trade_stats/2

    # Calculate max, mean, and min
    summary_stats = trade_stats.agg(['max', 'mean', 'min'])

    # Convert to dictionary
    stats_dict = {
        'max_trade_count': summary_stats.loc['max', 'trade_count'],
        'mean_trade_count': summary_stats.loc['mean', 'trade_count'],
        'min_trade_count': summary_stats.loc['min', 'trade_count'],
        'max_negative_trades': summary_stats.loc['max', 'negative_trades'],
        'mean_negative_trades': summary_stats.loc['mean', 'negative_trades'],
        'min_negative_trades': summary_stats.loc['min', 'negative_trades'],
        'max_positive_trades': summary_stats.loc['max', 'positive_trades'],
        'mean_positive_trades': summary_stats.loc['mean', 'positive_trades'],
        'min_positive_trades': summary_stats.loc['min', 'positive_trades'],
    }

    return stats_dict

    # return summary_stats.loc['max', 'trade_count'], summary_stats.loc['mean', 'trade_count'], summary_stats.loc['min', 'trade_count'], summary_stats.loc['max', 'negative_trades'], summary_stats.loc['mean', 'negative_trades'], summary_stats.loc['min', 'negative_trades'], summary_stats.loc['max', 'positive_trades'], summary_stats.loc['mean', 'positive_trades'], summary_stats.loc['min', 'positive_trades'],



def process_folder(prefix):
    main_folder_path = prefix
    print(main_folder_path)

    list_dicts = []

    # Iterate through each folder in the main folder
    folder_names = os.listdir(main_folder_path)
    folder_names.sort()
    for folder_name in folder_names:
        folder_path = os.path.join(main_folder_path, folder_name)
        if os.path.isdir(folder_path):
            print("Reading files from folder:", folder_name)
            # total_trades_per_folder = total_trades_folder(folder_path)
            # peak_contracts = peak_contracts_per_folder(folder_path)[0]
            # avg_contracts = peak_contracts_per_folder(folder_path)[1]

            total_trade_duration, avg_duration_per_trade,std_of_duration_per_trade ,pct_time_in_trade = time_in_trade(folder_path)
            # print('avg entry day: ', 22- round(avg_duration_per_trade/375))
            # print('std of entry day: ', round(std_of_duration_per_trade/375))
            
            exit_cooloff = folder_path.split('_')[-7]
            entry_cooloff = folder_path.split('_')[-5]

            period = folder_path.split('_')[-1]
            macd_variable = folder_path.split('_')[-3]

            # transaction_fee_blotter = get_transaction_fee_blotter(folder_path)
            transaction_fee_bt = get_transaction_fee(folder_path)

            stats_dict = get_trade_summary_stats(folder_path)

            trade_duration_stats = trade_duration(folder_path)

            # Iterate through files in the current folder
            for file_name in os.listdir(folder_path):
                if file_name.endswith('.xlsx'):
                    file_path = os.path.join(folder_path, file_name)
                
                    # Read the YAML file
                    excel_dict = excel_to_dict(file_path)
                    excel_dict['Year'] = str(folder_name.split('_')[0]) #str(int(folder_name.split('_')[0]))
                    # excel_dict['total_trades'] = str(total_trades_per_folder)
                    # excel_dict['peak_contracts'] = str(peak_contracts)
                    # excel_dict['avg_active_contracts'] = str(avg_contracts)
                    excel_dict['total_trade_duration'] = str(total_trade_duration)
                    excel_dict['avg_duration_per_trade'] = str(round(avg_duration_per_trade,2))
                    excel_dict['pct_time_in_trade'] = str(round(pct_time_in_trade,2))
                    excel_dict['entry_cooloff'] = entry_cooloff
                    excel_dict['exit_cooloff'] = exit_cooloff
                    excel_dict['macd_period'] = period
                    excel_dict['macd_variable'] = macd_variable
                    # excel_dict['transaction_fee_blotter'] = transaction_fee_blotter
                    # excel_dict['avg_lt_2min'] = trade_duration_stats[0]
                    # excel_dict['avg_lt_5min'] = trade_duration_stats[1]
                    # excel_dict['days_with_lt_2min'] = trade_duration_stats[2]
                    # excel_dict['days_with_lt_5min'] = trade_duration_stats[3]
                    # excel_dict['days_with_lt_2min_in_pct'] = trade_duration_stats[4]
                    # excel_dict['days_with_lt_5min_in_pct'] = trade_duration_stats[5]
                    excel_dict['transaction_fee_bt'] = transaction_fee_bt
                    converted_data = {key: convert_to_number(str(value)) for key, value in excel_dict.items()}
                    converted_data.update(stats_dict)
                    list_dicts.append(converted_data)
                    
        # break

    df = pd.DataFrame(list_dicts)
    df_selected = df[selected_columns]
    # print(df_selected)
    # return df_selected, main_folder_path
    df_selected.to_csv(os.path.join(main_folder_path,'output.csv'), index=False)

In [8]:
# path_list = glob.glob('/home/hsph/Code/EIS/new_backtest/HFT-Options-EIS-Global/tradelib/outputs/Dec9_sim/*')

In [3]:
# process_folder("/home/hsph/Code/EIS/new_backtest/HFT-Options-EIS-Global/tradelib/outputs/Jan27_sim/252_fix_daily_reset_rsi_cc_1_daily_entry_limit_1_no_intraday_rsi_win_15_ivBNF_cooloff_5_5/")
process_folder('/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/Condor/')
# for cool in range(30,90,10):
#     process_folder(f"/home/hsph/Code/EIS/new_backtest/HFT-Options-EIS-Global/tradelib/outputs/Dec19_sim/macd_vr_cc_1_daily_entry_limit_2BNF_cooloff_{90}_{cool}/")
# for path in path_list:
#     process_folder(path)

/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/Condor/
Reading files from folder: 2024_ASIANPAINT_STATIC_CONDOR_STRATEGY
Error: /home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/Condor/2024_ASIANPAINT_STATIC_CONDOR_STRATEGY
Reading files from folder: 2024_AXISBANK_STATIC_CONDOR_STRATEGY
Error: /home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/Condor/2024_AXISBANK_STATIC_CONDOR_STRATEGY
Reading files from folder: 2024_BHARTIARTL_STATIC_CONDOR_STRATEGY
Error: /home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/Condor/2024_BHARTIARTL_STATIC_CONDOR_STRATEGY
Reading files from folder: 2024_SBIN_STATIC_CONDOR_STRATEGY
Error: /home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/Condor/2024_SBIN_STATIC_CONDOR_STRATEGY


In [5]:

# path_to_backtest = '/home/hsph/Code/EIS/new_backtest/HFT-Options-EIS-Global/tradelib/outputs/Dec24_sim/macd_vr_cc_1_daily_entry_limit_2_delta_out_1.5BNF_cooloff_90_90/2023_vr_BANKNIFTY_MACD_SIGNAL_STRADDLE_STRATEGY_hedge_2_macd_period_3_10_30/'

# blotter_path_list = glob.glob(path_to_backtest + "blotter/*.csv")
# full_blotter_df = pd.concat((pd.read_csv(path) for path in blotter_path_list))
# full_blotter_df['date'] = pd.to_datetime(full_blotter_df['timestamp'])
# entry_dates = full_blotter_df.loc[full_blotter_df['trade_sub_type'] == 'trade', 'date'].unique()
# exit_dates = full_blotter_df.loc[full_blotter_df['trade_sub_type'] == 'unwind', 'date'].unique()

# entry_dates.sort()
# exit_dates.sort()

In [85]:
entry_dates.shape

(123,)

In [86]:
exit_dates.shape

(123,)

In [87]:
durations = []  


def count_weekends(start_date, end_date):
    # Generate all dates between start_date and end_date
    all_dates = np.arange(start_date, end_date + np.timedelta64(1, 'D'), dtype='datetime64[D]')
    days_of_week = (all_dates.astype('datetime64[D]') - np.datetime64('1970-01-05')).astype('timedelta64[D]').astype(int) % 7
    # Count Saturdays (5) and Sundays (6)
    weekends_count = np.isin(days_of_week, [5, 6]).sum()
    return weekends_count


for entry_date, exit_date in zip(entry_dates, exit_dates):
    # Calculate total difference in days
    entry_day = entry_date.astype('datetime64[D]')
    exit_day = exit_date.astype('datetime64[D]')
    total_days = (exit_day - entry_day).astype(int)
    weekend_days = count_weekends(entry_date, exit_date) 

    # Convert total days to minutes (18 hours per day = 1080 minutes)
    minutes_from_days = total_days * 1065 

    # Calculate total difference in minutes
    total_minutes = (exit_date - entry_date) / np.timedelta64(1, 'm')

    # Subtract minutes from days to get the remaining fractional minutes
    fractional_minutes = total_minutes - minutes_from_days - weekend_days * 375

    durations.append(fractional_minutes)

print("Durations (fractional minutes):", durations)
print("Total duration in minutes: ", sum(durations))
print("Average duration in minutes: ", sum(durations)/ len(durations))
print('time in trade pct: ', sum(durations)/(375*252) )



Durations (fractional minutes): [325.0, 274.0, 648.0, 275.0, 244.0, 2146.0, 1410.0, 267.0, 192.0, 382.0, 227.0, 226.0, 196.0, 263.0, 631.0, 648.0, 1504.0, 139.0, 263.0, 631.0, 234.0, 297.0, 169.0, 375.0, 641.0, 264.0, 219.0, 337.0, 892.0, 1003.0, 269.0, 258.0, 1047.0, 272.0, 653.0, 298.0, 258.0, 566.0, 2625.0, 1356.0, 250.0, 645.0, 253.0, 656.0, 259.0, 284.0, 177.0, 746.0, 643.0, 1404.0, 260.0, 648.0, 1021.0, 242.0, 651.0, 943.0, 349.0, 285.0, 270.0, 243.0, 228.0, 1013.0, 2872.0, 244.0, 1373.0, 576.0, 1024.0, 408.0, 249.0, 1032.0, 250.0, 1046.0, 583.0, 225.0, 1022.0, 281.0, 205.0, 635.0, 688.0, 1048.0, 1045.0, 235.0, 624.0, 1788.0, 293.0, 1373.0, 265.0, 170.0, 391.0, 307.0, 752.0, 1122.0, 2160.0, 1030.0, 946.0, 769.0, 251.0, 282.0, 1023.0, 1021.0, 483.0, 314.0, 631.0, 271.0, 1002.0, 185.0, 1115.0, 1006.0, 652.0, 246.0, 194.0, 2948.0, 571.0, 141.0, 737.0, 201.0, 360.0, 1726.0, 470.0, 2267.0, 990.0, 660.0, 178.0]
Total duration in minutes:  81350.0
Average duration in minutes:  661.38211

In [88]:
entry_date = entry_dates[-4]
exit_date = exit_dates[-4]

print(entry_date, exit_date)

weekend_days = count_weekends(entry_date, exit_date) 

print('weekdend days: ',weekend_days)

entry_day = entry_date.astype('datetime64[D]')
exit_day = exit_date.astype('datetime64[D]')

total_days = (exit_day - entry_day).astype(int)

print('total days: ', total_days)

# Convert total days to minutes (18 hours per day = 1080 minutes)
minutes_from_days = total_days * 1065 

print('minutes from days',minutes_from_days)

# Calculate total difference in minutes
total_minutes = (exit_date - entry_date) / np.timedelta64(1, 'm')

print('total mins: ',total_minutes)

# Subtract minutes from days to get the remaining fractional minutes
fractional_minutes = total_minutes - minutes_from_days - weekend_days * 375

print('frac mins: ',fractional_minutes)

2023-12-13T10:23:00.000000000 2023-12-21T10:40:00.000000000
weekdend days:  2
total days:  8
minutes from days 8520
total mins:  11537.0
frac mins:  2267.0


## Calculate total trades from blotter

In [6]:
blotter_df = pd.read_csv('/home/hsph/Downloads/fast_expiry/2021_BANKNIFTY_DAY_OF_MONTH_DUAL_SIGNAL_STATIC_CONDOR_STRATEGY/blotter/blotter_20210319092000_20210319153000.csv')

# filter only trades and negative position
trade_df = blotter_df[(blotter_df['trade_sub_type'] == 'trade') & (blotter_df['position'] < 0)]

# lot size / trade size
single_trade_position = trade_df['position'].values[0]

# dividing by 2 as put and call
total_trades = trade_df['position'].sum() / (single_trade_position*2)

In [7]:
total_trades_folder('/home/hsph/Downloads/fast_expiry/2021_BANKNIFTY_DAY_OF_MONTH_DUAL_SIGNAL_STATIC_CONDOR_STRATEGY/')

6732.0

In [3]:
folder_path = "/home/cloudcraftz/Office_Projects/10_Fintech/backtest_2.0_dev/HFT-Options-EIS-Global/tradelib/outputs/2024_NIFTYSTRADDLE_STRATEGY"

In [4]:
def get_transaction_fee(folder_path):
    try:
        Alpha_list = glob.glob(folder_path + '/consolidated_store/*.csv')
        Alpha_bt_df_list = [pd.read_csv(i) for i in Alpha_list]

        Alpha_df = pd.concat(Alpha_bt_df_list, ignore_index=True)
        Alpha_df = Alpha_df[Alpha_df['trade_done']==True]
        
        return Alpha_df['transaction_fees'].sum()
    except Exception as e:
        print(f"Error: {folder_path}")

In [5]:
get_transaction_fee(folder_path)

827346.6787500001